In [1]:
import os
os.chdir(r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11')

In [3]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [4]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [5]:
def CNN():
    input_data = Input(shape=(time_steps, num_features))
    x1 = Conv1D(16, 2, activation="relu")(input_data)
    x2 = Conv1D(16, 2, activation="relu")(x1)
    flatten = Flatten()(x2)
    output_data = Dense(1)(flatten)
    model = Model(input_data, output_data)
    return model

In [10]:
model1 = CNN()
model1.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 24, 21)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 23, 16)         │           688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 22, 16)         │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 352)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           353 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,569 (6.13 KB)

 Trainable params: 1,569 (6.13 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
!pip install pydot


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [12]:
checkpoints = r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

<>:3: SyntaxWarning: invalid escape sequence '\h'
<>:4: SyntaxWarning: invalid escape sequence '\h'
<>:3: SyntaxWarning: invalid escape sequence '\h'
<>:4: SyntaxWarning: invalid escape sequence '\h'
C:\Users\Muhammad Mubien\AppData\Local\Temp\ipykernel_13972\114443299.py:3: SyntaxWarning: invalid escape sequence '\h'
  FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
C:\Users\Muhammad Mubien\AppData\Local\Temp\ipykernel_13972\114443299.py:4: SyntaxWarning: invalid escape sequence '\h'
  JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])


In [13]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [14]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =CNN()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [19]:
import os
path_dataset =r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEPscaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Muhammad Mubien\Documents\python env 3.12\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [20]:
time_steps=24
num_features=21

In [21]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.007993936538696289 sec


In [22]:
epochs = 60
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/60
21/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2291 - mae: 0.2291 - mape: 118.8895
Epoch 1: val_loss improved from None to 0.07505, saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0001-loss0.08.h5



Epoch 1: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0001-loss0.08.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - loss: 0.1401 - mae: 0.1401 - mape: 88.3339 - val_loss: 0.0751 - val_mae: 0.0751 - val_mape: 27.1682
Epoch 2/60
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0754 - mae: 0.0754 - mape: 44.7986
Epoch 2: val_loss improved from 0.07505 to 0.05796, saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0002-loss0.06.h5



Epoch 2: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0002-loss0.06.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0736 - mae: 0.0736 - mape: 45.2389 - val_loss: 0.0580 - val_mae: 0.0580 - val_mape: 16.6455
Epoch 3/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0600 - mae: 0.0600 - mape: 35.9659
Epoch 3: val_loss improved from 0.05796 to 0.05574, saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0003-loss0.06.h5



Epoch 3: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0003-loss0.06.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0602 - mae: 0.0602 - mape: 38.4941 - val_loss: 0.0557 - val_mae: 0.0557 - val_mape: 17.1458
Epoch 4/60
20/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0617 - mae: 0.0617 - mape: 35.5120  
Epoch 4: val_loss did not improve from 0.05574
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0610 - mae: 0.0610 - mape: 32.1739 - val_loss: 0.0643 - val_mae: 0.0643 - val_mape: 19.4144
Epoch 5/60
19/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0547 - mae: 0.0547 - mape: 25.8292
Epoch 5: val_loss improved from 0.05574 to 0.05403, saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0005-loss0.05.h5



Epoch 5: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0005-loss0.05.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0539 - mae: 0.0539 - mape: 30.0598 - val_loss: 0.0540 - val_mae: 0.0540 - val_mape: 17.8267
Epoch 6/60
26/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0587 - mae: 0.0587 - mape: 27.2888
Epoch 6: val_loss improved from 0.05403 to 0.03838, saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0006-loss0.04.h5



Epoch 6: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0006-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 0.0557 - mae: 0.0557 - mape: 26.8068 - val_loss: 0.0384 - val_mae: 0.0384 - val_mape: 11.6033
Epoch 7/60
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0518 - mae: 0.0518 - mape: 26.3953
Epoch 7: val_loss did not improve from 0.03838
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0487 - mae: 0.0487 - mape: 23.8692 - val_loss: 0.0550 - val_mae: 0.0550 - val_mape: 15.3424
Epoch 8/60
26/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0435 - mae: 0.0435 - mape: 23.4326 
Epoch 8: val_loss did not improve from 0.03838
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.0427 - mae: 0.0427 - mape: 23.8827 - val_loss: 0.0466 - val_mae: 0.0466 - val_mape: 14.6569
Epoch 9/60
26/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0424 - mae: 0.0424 - mape: 19.4918
Epoch 9: val_loss did not improve from 0.03838
27/27 ━━━━━━━━━━━━━━━━━━━


Epoch 23: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0023-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 0.0274 - mae: 0.0274 - mape: 11.2416 - val_loss: 0.0352 - val_mae: 0.0352 - val_mape: 11.8626
Epoch 24/60
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0267 - mae: 0.0267 - mape: 13.6233
Epoch 24: val_loss did not improve from 0.03516
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.0268 - mae: 0.0268 - mape: 12.8595 - val_loss: 0.0361 - val_mae: 0.0361 - val_mape: 11.7887
Epoch 25/60
23/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0241 - mae: 0.0241 - mape: 13.3216
Epoch 25: val_loss did not improve from 0.03516
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.0251 - mae: 0.0251 - mape: 11.3298 - val_loss: 0.0432 - val_mae: 0.0432 - val_mape: 15.3708
Epoch 26/60
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0283 - mae: 0.0283 - mape: 13.3950
Epoch 26: val_loss did not improve from 0.03516
27/27 ━━━━━━━━━━━━━


Epoch 46: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0046-loss0.03.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0215 - mae: 0.0215 - mape: 9.3665 - val_loss: 0.0348 - val_mae: 0.0348 - val_mape: 11.7795
Epoch 47/60
21/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0193 - mae: 0.0193 - mape: 8.6606  
Epoch 47: val_loss did not improve from 0.03479
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0198 - mae: 0.0198 - mape: 8.8119 - val_loss: 0.0402 - val_mae: 0.0402 - val_mape: 14.0143
Epoch 48/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0213 - mae: 0.0213 - mape: 9.7390
Epoch 48: val_loss did not improve from 0.03479
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0207 - mae: 0.0207 - mape: 9.5821 - val_loss: 0.0415 - val_mae: 0.0415 - val_mape: 13.9735
Epoch 49/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0205 - mae: 0.0205 - mape: 8.1000
Epoch 49: val_loss did not improve from 0.03479
27/27 ━━━━━━━━━━━━━━━━━

In [25]:

model = load_model(
    r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0023-loss0.04.h5',
    compile=False
)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step
Mean Absolute Error (MAE): 7126.31
Median Absolute Error (MedAE): 7397.05
Mean Squared Error (MSE): 51575872.37
Root Mean Squared Error (RMSE): 7181.63
Mean Absolute Percentage Error (MAPE): 45.62 %
Median Absolute Percentage Error (MDAPE): 47.71 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


In [30]:
checkpoints = r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\E1-cp-0023-loss0.04.h5'
start_epoch= 58

In [36]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
model = load_model(
    r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0023-loss0.04.h5',
    compile=False
)
    # update the learning rate
opt = Adam(1e-3)
model.compile(loss='mae', optimizer=opt, metrics=["mae", "mape"])
y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)

[INFO] loading <Functional name=functional_2, built=True>...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step


In [37]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
21/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0414 - mae: 0.0414 - mape: 18.4988
Epoch 1: val_loss improved from None to 0.04912, saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\E2-cp-0001-loss0.05.h5



Epoch 1: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\E2-cp-0001-loss0.05.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - loss: 0.0313 - mae: 0.0313 - mape: 13.8682 - val_loss: 0.0491 - val_mae: 0.0491 - val_mape: 16.6887
Epoch 2/10
23/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0270 - mae: 0.0270 - mape: 13.6902
Epoch 2: val_loss improved from 0.04912 to 0.04638, saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\E2-cp-0002-loss0.05.h5



Epoch 2: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\E2-cp-0002-loss0.05.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0266 - mae: 0.0266 - mape: 13.5736 - val_loss: 0.0464 - val_mae: 0.0464 - val_mape: 15.8430
Epoch 3/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0294 - mae: 0.0294 - mape: 10.8017
Epoch 3: val_loss improved from 0.04638 to 0.03773, saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\E2-cp-0003-loss0.04.h5



Epoch 3: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\E2-cp-0003-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0265 - mae: 0.0265 - mape: 11.1728 - val_loss: 0.0377 - val_mae: 0.0377 - val_mape: 12.1331
Epoch 4/10
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0270 - mae: 0.0270 - mape: 13.9821
Epoch 4: val_loss did not improve from 0.03773
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.0253 - mae: 0.0253 - mape: 11.2818 - val_loss: 0.0411 - val_mae: 0.0411 - val_mape: 13.5246
Epoch 5/10
19/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0265 - mae: 0.0265 - mape: 11.1486
Epoch 5: val_loss did not improve from 0.03773
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.0247 - mae: 0.0247 - mape: 11.7435 - val_loss: 0.0490 - val_mae: 0.0490 - val_mape: 16.2732
Epoch 6/10
20/27 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0346 - mae: 0.0346 - mape: 14.6018
Epoch 6: val_loss did not improve from 0.03773
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 2


Epoch 10: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\E2-cp-0010-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0279 - mae: 0.0279 - mape: 12.2931 - val_loss: 0.0353 - val_mae: 0.0353 - val_mape: 11.6365


In [39]:

model = load_model(
    r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab11\delete\E1-cp-0023-loss0.04.h5',
    compile=False
)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step
Mean Absolute Error (MAE): 7126.31
Median Absolute Error (MedAE): 7397.05
Mean Squared Error (MSE): 51575872.37
Root Mean Squared Error (RMSE): 7181.63
Mean Absolute Percentage Error (MAPE): 45.62 %
Median Absolute Percentage Error (MDAPE): 47.71 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


# lab report 

## Lab 1